# LEO nanosatellite TinyML triage: MobileNetV3-small -> structured L2 pruning -> INT8 (ONNX Runtime)
Run the cells top to bottom (Runtime > Change runtime type > T4 GPU recommended; CPU works but is slow).
Optional inputs: `/content/cache.npz` (38-Cloud tiles) and your own satellite RGB images in `/content/sat_images/`.
Things I could not verify (no Colab/GPU available while writing) are listed in the last cell.

In [ ]:
# 1) Install dependencies (torch/torchvision are preinstalled on Colab).
!pip -q install torch-pruning onnx onnxruntime onnxscript scikit-learn pandas pillow scipy

In [ ]:
# 2) Imports, seeds, global config constants.
import os, io, json, time, math, copy, random, glob, zipfile, shutil, warnings
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
from PIL import Image, ImageDraw
warnings.filterwarnings("ignore")

SEED = 0
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '| torch', torch.__version__)

S = 160                                   # tile size fed to the network
CLASS_NAMES = ['AnnualCrop','Forest','HerbaceousVegetation','Highway','Industrial',
               'Pasture','PermanentCrop','Residential','River','SeaLake']
W_CLOUD, W_FRAC, W_CLS = 1.0, 1.0, 1.0    # loss weights
BS = 128                                  # batch size
EPOCH_STEPS = 120                         # optimisation steps per "epoch" (lower = faster)
MAX_DROP = 2.0                            # allowed accuracy drop (points) on BOTH heads
SALIENCE = {'Industrial':1.0,'Residential':0.9,'Highway':0.8,'River':0.7,'SeaLake':0.5,'AnnualCrop':0.5,
            'PermanentCrop':0.5,'Pasture':0.4,'HerbaceousVegetation':0.3,'Forest':0.3}  # EDIT ME

In [ ]:
# 3) Load EuroSAT RGB, stratified 80/10/10 split, optional real 38-Cloud tiles split BY SCENE.
ds = torchvision.datasets.EuroSAT('/content/eurosat', download=True)
assert list(ds.classes) == CLASS_NAMES, ds.classes
X = np.stack([np.asarray(Image.open(p).convert('RGB')) for p, _ in ds.samples])   # (27000,64,64,3) uint8
Y = np.array(ds.targets)
idx = np.arange(len(Y))
tr_i, tmp_i = train_test_split(idx, test_size=0.2, stratify=Y, random_state=SEED)
va_i, te_i = train_test_split(tmp_i, test_size=0.5, stratify=Y[tmp_i], random_state=SEED)
print('EuroSAT split', len(tr_i), len(va_i), len(te_i))
T = lambda a, dt=None: torch.as_tensor(a, device=dev) if dt is None else torch.as_tensor(a, device=dev).to(dt)
Xtr, Ytr = T(X[tr_i]), T(Y[tr_i]).long()           # uint8 images live on the GPU (small)
Xva, Yva = T(X[va_i]), T(Y[va_i]).long()
Xte, Yte = T(X[te_i]), T(Y[te_i]).long()

REAL = os.path.exists('/content/cache.npz')
Rtr = FRtr = Rva = FRva = Rte = FRte = None
if REAL:
    d = np.load('/content/cache.npz')
    Xr, Yr, sc = d['X'], d['Y'], d['scene']
    fr_all = (Yr > 0).reshape(len(Yr), -1).mean(1).astype(np.float32)      # cloud_frac = mask mean
    uniq = np.unique(sc); hold = np.random.RandomState(SEED).permutation(uniq)
    val_sc, te_sc = hold[:4], hold[4:6]               # 4 scenes for validation (+2 extra scenes kept for test)
    mv, mt = np.isin(sc, val_sc), np.isin(sc, te_sc); mtr = ~(mv | mt)
    Rtr, FRtr = T(Xr[mtr]), T(fr_all[mtr])
    Rva, FRva = T(Xr[mv]), T(fr_all[mv])
    Rte, FRte = T(Xr[mt]), T(fr_all[mt])
    print('real tiles train/val/test:', int(mtr.sum()), int(mv.sum()), int(mt.sum()))
else:
    print('No /content/cache.npz -> synthetic clouds only.')

In [ ]:
# 4) GPU data synthesis: fractal clouds (thick + cirrus, soft edges), snow/bright-surface hard negatives, augmentations.
def to160(xu8):                                   # uint8 (B,h,w,3) -> float (B,3,S,S) in 0-1
    x = xu8.permute(0, 3, 1, 2).float() / 255
    return F.interpolate(x, size=(S, S), mode='bilinear', align_corners=False)

def fractal(B, H, W, octaves=(4, 8, 16, 32, 64), pers=0.55):   # multi-scale value noise normalised to 0-1
    n, amp = 0, 1.0
    for o in octaves:
        g = (max(2, round(o * H / 160)), max(2, round(o * W / 160)))
        n = n + amp * F.interpolate(torch.rand(B, 1, *g, device=dev), size=(H, W), mode='bicubic', align_corners=False)
        amp *= pers
    mn, mx = n.amin((1, 2, 3), keepdim=True), n.amax((1, 2, 3), keepdim=True)
    return (n - mn) / (mx - mn + 1e-6)

def add_clouds(x, clean_p=0.5, octaves=(4, 8, 16, 32, 64), snow_p=0.25):
    """Blend clouds (target coverage ~U(0,1)); ~clean_p stay clean; some clean ones get snow/bright patches. Returns x, cloud_frac."""
    B, _, H, W = x.shape; d = x.device
    cov = torch.rand(B, device=d); cov[torch.rand(B, device=d) < clean_p] = 0
    n = fractal(B, H, W, octaves)
    ns = n.flatten(1).sort(1).values
    thr = ns.gather(1, ((1 - cov) * (H * W - 1)).long().clamp(0, H * W - 1)[:, None]).view(B, 1, 1, 1)
    a = ((n - thr) / 0.10).clamp(0, 1)                                    # soft edges
    thin = torch.rand(B, 1, 1, 1, device=d) < 0.3                          # 30% thin semi-transparent cirrus
    op = torch.where(thin, 0.3 + 0.4 * torch.rand(B, 1, 1, 1, device=d), 0.85 + 0.15 * torch.rand(B, 1, 1, 1, device=d))
    alpha = a * op * (cov > 0).view(B, 1, 1, 1).float()
    det = fractal(B, H, W, (16, 32, 64, 128))
    g = 0.8 + 0.2 * torch.rand(B, 1, 1, 1, device=d)                      # white .. light grey
    col = (g * (0.9 + 0.1 * det)).expand(-1, 3, -1, -1)
    x = x * (1 - alpha) + col * alpha
    frac = (alpha > 0.4).float().mean((1, 2, 3))                          # label = fraction with alpha>0.4
    sn = (cov == 0) & (torch.rand(B, device=d) < snow_p)                  # hard negatives: snow / salt / bright sand
    if sn.any():
        k = int(sn.sum()); nz = fractal(k, H, W, (2, 4, 8, 16))
        c = 0.15 + 0.65 * torch.rand(k, device=d)
        th = nz.flatten(1).sort(1).values.gather(1, ((1 - c) * (H * W - 1)).long()[:, None]).view(k, 1, 1, 1)
        mask = (nz > th).float()                                          # HARD edge (no soft blending)
        fine = F.interpolate(torch.rand(k, 1, H // 2, W // 2, device=d), size=(H, W), mode='nearest')
        tint = torch.where(torch.rand(k, 1, 1, 1, device=d) < 0.5,
                           torch.tensor([0.93, 0.96, 1.0], device=d).view(1, 3, 1, 1),
                           torch.tensor([0.95, 0.92, 0.84], device=d).view(1, 3, 1, 1))
        x = x.clone(); x[sn] = x[sn] * (1 - mask) + tint * (0.8 + 0.2 * fine) * mask
    return x.clamp(0, 1), frac

def augment(x, zoom_in=True):                    # flips, rot90, zoom 0.7-1.4, brightness/contrast/gamma/saturation
    B = x.shape[0]; d = x.device; r = lambda *s: torch.rand(*s, device=d)
    x = torch.where(r(B, 1, 1, 1) < .5, x.flip(3), x); x = torch.where(r(B, 1, 1, 1) < .5, x.flip(2), x)
    x = torch.rot90(x, random.randint(0, 3), (2, 3))
    outs = []
    for c in x.chunk(4):                         # each chunk gets its own zoom factor
        z = random.uniform(0.7, 1.4)
        if z < 1:                                # zoom out = coarser ground resolution (down then up)
            c = F.interpolate(F.interpolate(c, size=max(16, int(S * z)), mode='bilinear', align_corners=False), size=S, mode='bilinear', align_corners=False)
        elif zoom_in:                            # zoom in = finer resolution (crop then up); skipped for real tiles (mask frac must stay valid)
            cs = int(S / z); o = (S - cs) // 2
            c = F.interpolate(c[:, :, o:o + cs, o:o + cs], size=S, mode='bilinear', align_corners=False)
        outs.append(c)
    x = torch.cat(outs)
    g, ct, b, s = 0.7 + 0.7 * r(B, 1, 1, 1), 0.75 + 0.55 * r(B, 1, 1, 1), (r(B, 1, 1, 1) - .5) * 0.3, 0.6 + 0.8 * r(B, 1, 1, 1)
    x = x.clamp(1e-4, 1) ** g
    m = x.mean((1, 2, 3), keepdim=True); x = (x - m) * ct + m + b
    gr = x.mean(1, keepdim=True); x = gr + (x - gr) * s
    return x.clamp(0, 1)

def sensor(x):                                   # slight blur + noise applied AFTER clouds (sensor effects)
    B = x.shape[0]
    m = (torch.rand(B, 1, 1, 1, device=x.device) < 0.3).float()
    x = m * F.avg_pool2d(x, 3, 1, 1) + (1 - m) * x
    return (x + torch.randn_like(x) * torch.rand(B, 1, 1, 1, device=x.device) * 0.03).clamp(0, 1)

# quick visual sanity check of the synthesis
_x, _f = add_clouds(to160(Xtr[:8]), clean_p=0.0)
_g = torch.cat([to160(Xtr[:8]), _x], 0)
torchvision.utils.save_image(_g, '/content/synth_check.png', nrow=8)
print('synthetic cloud fracs:', np.round(_f.cpu().numpy(), 2), '-> see /content/synth_check.png')

In [ ]:
# 5) Batch builder (EuroSAT + optional real tiles) and fixed validation/test sets.
def build_batch(xe=None, ye=None, xr=None, fr=None, train=True):
    xs, yf, yk = [], [], []
    if xe is not None and len(xe):
        x = to160(xe)
        if train: x = augment(x, True)
        x, f = add_clouds(x); xs.append(x); yf.append(f); yk.append(ye.long())
    if xr is not None and len(xr):
        x = to160(xr)
        if train: x = augment(x, False)
        xs.append(x); yf.append(fr.float()); yk.append(torch.full((len(xr),), -1, device=x.device, dtype=torch.long))
    x, f, k = torch.cat(xs), torch.cat(yf), torch.cat(yk)
    if train: x = sensor(x)
    return x.clamp(0, 1), (f >= 0.5).float(), f, k       # cloudy = frac >= 0.5 ; class label -1 => masked

def train_batch():
    nr = BS // 4 if REAL else 0
    i = torch.randint(0, len(Xtr), (BS - nr,), device=dev)
    if nr:
        j = torch.randint(0, len(Rtr), (nr,), device=dev)
        return build_batch(Xtr[i], Ytr[i], Rtr[j], FRtr[j], True)
    return build_batch(Xtr[i], Ytr[i], train=True)

def make_eval(xe, ye, xr=None, fr=None, seed=0):    # fixed (seeded) eval set stored as uint8 on CPU
    seed_all(seed); L = {'x': [], 'yc': [], 'yf': [], 'yk': []}
    def add(b):
        x, c, f, k = b
        L['x'].append((x * 255).round().byte().cpu()); L['yc'].append(c.cpu()); L['yf'].append(f.cpu()); L['yk'].append(k.cpu())
    for i in range(0, len(xe), 512): add(build_batch(xe[i:i+512], ye[i:i+512], train=False))
    if xr is not None:
        for i in range(0, len(xr), 512): add(build_batch(xr=xr[i:i+512], fr=fr[i:i+512], train=False))
    return {'x': torch.cat(L['x']), 'yc': torch.cat(L['yc']).long(), 'yf': torch.cat(L['yf']), 'yk': torch.cat(L['yk']).long()}

VAL = make_eval(Xva, Yva, Rva, FRva, seed=1)
TEST = make_eval(Xte, Yte, Rte, FRte, seed=2)
print('VAL', VAL['x'].shape, 'cloudy %.2f' % VAL['yc'].float().mean(), '| TEST', TEST['x'].shape)

In [ ]:
# 6) Model, loss, metrics, and a generic fit() with early stopping.
class Net(nn.Module):
    def __init__(s):
        super().__init__()
        m = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        s.features = m.features; s.pool = nn.AdaptiveAvgPool2d(1); D = 576
        s.cloud = nn.Sequential(nn.Linear(D, 64), nn.ReLU(), nn.Linear(64, 2))                          # cloud_logit, cloud_frac_logit
        s.cls = nn.Sequential(nn.Linear(D, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 10))       # 10 EuroSAT classes
        s.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        s.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(s, x):                                   # x: raw 0-1 RGB; normalisation is INSIDE the model
        e = torch.flatten(s.pool(s.features((x - s.mean) / s.std)), 1)
        c = s.cloud(e)
        return c[:, 0:1], c[:, 1:2], s.cls(e), e

def loss_fn(o, yc, yf, yk):
    cl, fl, ck, _ = o
    l1 = F.binary_cross_entropy_with_logits(cl[:, 0], yc)
    l2 = F.l1_loss(torch.sigmoid(fl[:, 0]), yf)
    m = yk >= 0
    l3 = F.cross_entropy(ck[m], yk[m]) if m.any() else cl.sum() * 0
    return W_CLOUD * l1 + W_FRAC * l2 + W_CLS * l3

@torch.no_grad()
def predict_fp(model, xu8, bs=256):
    model.eval().to(dev); outs = [[], [], [], []]
    for i in range(0, len(xu8), bs):
        o = model(xu8[i:i+bs].to(dev).float() / 255)
        for k in range(4): outs[k].append(o[k].float().cpu().numpy())
    return [np.concatenate(o) for o in outs]

@torch.no_grad()
def val_loss(model):
    model.eval().to(dev); tot = 0
    for i in range(0, len(VAL['x']), 512):
        o = model(VAL['x'][i:i+512].to(dev).float() / 255)
        tot += float(loss_fn(o, VAL['yc'][i:i+512].to(dev).float(), VAL['yf'][i:i+512].to(dev), VAL['yk'][i:i+512].to(dev))) * len(o[0])
    return tot / len(VAL['x'])

sig = lambda z: 1 / (1 + np.exp(-z))
def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def score(o, ev, Tc=1., Tk=1.):                         # all metrics for one set of logits
    cl, fl, ck, _ = o; yc = ev['yc'].numpy(); yf = ev['yf'].numpy(); yk = ev['yk'].numpy()
    p = sig(cl[:, 0] / Tc); pred = (p > 0.5).astype(int); cc = np.maximum(p, 1 - p); hc = cc >= 0.95
    r = {'cloud_acc': (pred == yc).mean() * 100, 'cloud_F1': f1_score(yc, pred) * 100,
         'cloud_AUC': roc_auc_score(yc, p) * 100 if len(np.unique(yc)) > 1 else np.nan,
         'cloud_frac_MAE': np.abs(sig(fl[:, 0]) - yf).mean(),
         'cloud_sel_acc@.95': (pred[hc] == yc[hc]).mean() * 100 if hc.any() else np.nan, 'cloud_cov@.95': hc.mean() * 100}
    m = yk >= 0; pr = softmax(ck[m] / Tk); pk = pr.argmax(1); yy = yk[m]; hi = pr.max(1) >= 0.95
    r.update({'cls_top1': (pk == yy).mean() * 100, 'cls_macroF1': f1_score(yy, pk, average='macro') * 100,
              'cls_sel_acc@.95': (pk[hi] == yy[hi]).mean() * 100 if hi.any() else np.nan, 'cls_cov@.95': hi.mean() * 100,
              'cm': confusion_matrix(yy, pk, labels=range(10))})
    return r

def fit(model, epochs, lr_b, lr_h, freeze=False, patience=4, tag=''):
    model.to(dev)
    bb = list(model.features.parameters()); hd = list(model.cloud.parameters()) + list(model.cls.parameters())
    for p in bb: p.requires_grad = not freeze
    opt = torch.optim.AdamW([{'params': bb, 'lr': lr_b}, {'params': hd, 'lr': lr_h}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * EPOCH_STEPS)
    best, bad = (1e9, None), 0
    for ep in range(epochs):
        model.train()
        if freeze: model.features.eval()                 # keep BN statistics fixed while frozen
        for _ in range(EPOCH_STEPS):
            x, yc, yf, yk = train_batch()
            loss = loss_fn(model(x), yc, yf, yk)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        vl = val_loss(model); print(f'{tag} ep{ep+1}/{epochs} train_loss {float(loss):.3f} val_loss {vl:.3f}')
        if vl < best[0]: best, bad = (vl, copy.deepcopy(model.state_dict())), 0
        else:
            bad += 1
            if bad >= patience: print('early stop'); break
    model.load_state_dict(best[1]); return model

In [ ]:
# 7) Train the FP32 baseline (phase 1: frozen backbone, phase 2: full fine-tune) and report metrics.
seed_all()
base = Net().to(dev)
fit(base, 3, 0, 1e-3, freeze=True, tag='P1')
fit(base, 15, 3e-4, 1e-3, freeze=False, patience=5, tag='P2')
base_val = score(predict_fp(base, VAL['x']), VAL)
base_test = score(predict_fp(base, TEST['x']), TEST)
torch.save(base.state_dict(), '/content/base_fp32.pt')
print({k: round(float(v), 3) for k, v in base_test.items() if k != 'cm'})
print(pd.DataFrame(base_test['cm'], index=CLASS_NAMES, columns=[c[:5] for c in CLASS_NAMES]))

In [ ]:
# 8) Iterative structured L2-norm filter pruning (torch-pruning, 5 steps, 3 fine-tune epochs per step).
import torch_pruning as tp
class PrunWrap(nn.Module):                               # tuple outputs -> ONE concatenated tensor for the dependency tracer
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, x): o = s.m(x); return torch.cat([o[0], o[1], o[2]], 1)

pruned = copy.deepcopy(base).cpu().eval()
wrap, ex = PrunWrap(pruned), torch.randn(1, 3, S, S)
ignored = [pruned.cloud[0], pruned.cloud[2], pruned.cls[0], pruned.cls[3]]   # head layers keep their output sizes
imp = tp.importance.MagnitudeImportance(p=2)                                  # L2-norm filter importance
kw = dict(importance=imp, iterative_steps=5, ignored_layers=ignored)
try:    pruner = tp.pruner.MetaPruner(wrap, ex, pruning_ratio=0.5, **kw)      # torch-pruning >= 1.3
except TypeError: pruner = tp.pruner.MetaPruner(wrap, ex, ch_sparsity=0.5, **kw)   # older API
macs0, par0 = tp.utils.count_ops_and_params(wrap, ex)
print(f'baseline: {par0/1e6:.2f}M params, {macs0/1e6:.1f}M MACs')

keep, hist = (0, copy.deepcopy(base).cpu()), []
for step in range(1, 6):
    pruned.cpu().eval(); pruner.step()                                        # remove the next slice of low-L2 filters
    fit(pruned, 3, 2e-4, 5e-4, patience=10, tag=f'prune{step}')
    r = score(predict_fp(pruned, VAL['x']), VAL); pruned.cpu()
    dc, dk = base_val['cloud_acc'] - r['cloud_acc'], base_val['cls_top1'] - r['cls_top1']
    macs, par = tp.utils.count_ops_and_params(PrunWrap(pruned), ex)
    ok = dc < MAX_DROP and dk < MAX_DROP
    hist.append(dict(step=step, params_M=par/1e6, MMACs=macs/1e6, drop_cloud=dc, drop_cls=dk, ok=ok))
    print(hist[-1])
    if ok: keep = (step, copy.deepcopy(pruned).cpu())                         # keep the SMALLEST model within budget
print(pd.DataFrame(hist).round(3).to_string(index=False))
print('selected pruning step:', keep[0], '(0 = no pruned model met the <2 point rule on both heads)')
pruned = keep[1]

In [ ]:
# 9) ONNX export, static INT8 (QDQ, per-channel) quantization, estimation helpers; automatic ReLU6 fallback if INT8 hurts.
import onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat, CalibrationMethod
OUT = ['cloud_logit', 'cloud_frac_logit', 'class_logits', 'embedding']

def export_onnx(model, path, dynamic=True):             # opset 13, input 'x', 4 named outputs, dynamic batch axis
    model.eval().cpu(); d = torch.rand(1, 3, S, S)
    dyn = {'dynamic_axes': {'x': {0: 'batch'}, **{n: {0: 'batch'} for n in OUT}}} if dynamic else {}
    try: torch.onnx.export(model, d, path, input_names=['x'], output_names=OUT, opset_version=13, dynamo=False, **dyn)
    except TypeError: torch.onnx.export(model, d, path, input_names=['x'], output_names=OUT, opset_version=13, **dyn)

# calibration set: ~300 training tiles (cloudy + clear + snow), no augmentation
seed_all(7)
_i = torch.randperm(len(Xtr), device=dev)[:250]
if REAL: _j = torch.randperm(len(Rtr), device=dev)[:50]; _b = build_batch(Xtr[_i], Ytr[_i], Rtr[_j], FRtr[_j], train=False)
else:    _b = build_batch(Xtr[_i], Ytr[_i], train=False)
CAL = _b[0].cpu().numpy().astype(np.float32)
print('calibration tiles:', CAL.shape, 'cloudy share %.2f' % _b[1].mean())

class CalR(CalibrationDataReader):
    def __init__(s, x, bs=10): s.it = iter([{'x': x[i:i+bs]} for i in range(0, len(x), bs)])
    def get_next(s): return next(s.it, None)

def quantize(fp, q, method):
    pre = fp.replace('.onnx', '_pre.onnx')
    try:
        from onnxruntime.quantization.shape_inference import quant_pre_process
        quant_pre_process(fp, pre, skip_symbolic_shape=True)
    except Exception as e: print('  (quant_pre_process skipped:', str(e)[:80], ')'); pre = fp
    quantize_static(pre, q, CalR(CAL), quant_format=QuantFormat.QDQ, per_channel=True,
                    weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8, calibrate_method=method)

def ort_sess(path, threads=None):
    so = ort.SessionOptions()
    if threads: so.intra_op_num_threads = threads
    return ort.InferenceSession(path, so, providers=['CPUExecutionProvider'])

def predict_ort(sess, xu8, bs=64):
    outs = [[] for _ in OUT]
    for i in range(0, len(xu8), bs):
        o = sess.run(OUT, {'x': (xu8[i:i+bs].float() / 255).numpy().astype(np.float32)})
        for k in range(4): outs[k].append(o[k])
    return [np.concatenate(o) for o in outs]

def latency_ms(path, n=100):                            # single tile, 1 CPU thread
    s = ort_sess(path, 1); x = np.random.rand(1, 3, S, S).astype(np.float32)
    for _ in range(10): s.run(OUT, {'x': x})
    t = []
    for _ in range(n): t0 = time.perf_counter(); s.run(OUT, {'x': x}); t.append(time.perf_counter() - t0)
    return float(np.median(t) * 1000)

def peak_act_elems(model):
    """Peak activation memory (in ELEMENTS) for one 160x160 tile: for every node, sum the sizes of all tensors that are
    live at that point (the node's inputs + outputs + any earlier tensors still needed later, e.g. residual skips),
    take the max over nodes. Weights excluded; no in-place/fusion tricks assumed (conservative)."""
    p = '/content/_static.onnx'; export_onnx(model, p, dynamic=False)
    g = onnx.shape_inference.infer_shapes(onnx.load(p)).graph
    init = {i.name for i in g.initializer}; size = {}
    for v in list(g.value_info) + list(g.input) + list(g.output):
        dims = [d.dim_value for d in v.type.tensor_type.shape.dim]
        if v.name not in init and dims and all(d > 0 for d in dims): size[v.name] = int(np.prod(dims))
    birth = {n: -1 for n in size if n == 'x'}; last = {}
    for k, nd in enumerate(g.node):
        if nd.op_type == 'Constant': continue
        for o in nd.output: birth[o] = k
        for i in nd.input: last[i] = k
    for o in g.output: last[o.name] = len(g.node)
    return max(sum(sz for t, sz in size.items() if t in birth and birth[t] <= k <= last.get(t, birth[t])) for k in range(len(g.node)))

def drops(r): return base_val['cloud_acc'] - r['cloud_acc'], base_val['cls_top1'] - r['cls_top1']

def build_int8(model, tag):                             # export -> try MinMax then Percentile calibration -> keep best on VAL
    fp = f'/content/{tag}_fp32.onnx'; export_onnx(model, fp); best = None
    for meth in (CalibrationMethod.MinMax, CalibrationMethod.Percentile):
        q = f'/content/{tag}_{meth.name}_int8.onnx'
        try: quantize(fp, q, meth)
        except Exception as e: print('quantization failed for', meth.name, ':', str(e)[:120]); continue
        r = score(predict_ort(ort_sess(q), VAL['x']), VAL); w = max(drops(r))
        print(f'{tag} INT8 {meth.name}: worst head drop {w:.2f} pts (cloud {drops(r)[0]:.2f}, cls {drops(r)[1]:.2f})')
        if best is None or w < best[0]: best = (w, q, r, meth.name)
        if w < MAX_DROP: break
    assert best is not None, 'all quantization attempts failed - see troubleshooting notes'
    return fp, best

def swap_hswish(m):                                     # Hardswish -> ReLU6 (INT8-friendlier); Hardsigmoid in SE kept (0-1 gate)
    for n, c in m.named_children():
        if isinstance(c, nn.Hardswish): setattr(m, n, nn.ReLU6())
        else: swap_hswish(c)

fp_path, best = build_int8(pruned, 'pruned')
ACT_NOTE = 'Hardswish kept (INT8 within the 2-point budget)'
if best[0] >= MAX_DROP:
    print('INT8 drop >= 2 pts -> replacing Hardswish with ReLU6 and fine-tuning 4 epochs, then re-quantizing')
    swap_hswish(pruned); fit(pruned, 4, 2e-4, 5e-4, patience=10, tag='relu6-ft')
    fp_path, best = build_int8(pruned, 'relu6')
    ACT_NOTE = 'Hardswish replaced by ReLU6 + 4 epoch fine-tune (Hardswish destabilised INT8 activations)'
shutil.copy(best[1], '/content/model_int8.onnx'); FINAL_FP = pruned
print('final INT8 model uses calibration:', best[3], '|', ACT_NOTE)

In [ ]:
# 10) Final comparison table on the TEST set: FP32 vs pruned FP32 vs INT8.
export_onnx(base, '/content/base_fp32.onnx')
export_onnx(FINAL_FP, '/content/pruned_fp32.onnx')
paths = {'FP32': '/content/base_fp32.onnx', 'Pruned FP32': '/content/pruned_fp32.onnx', 'INT8': '/content/model_int8.onnx'}
res = {'FP32': base_test, 'Pruned FP32': score(predict_fp(FINAL_FP, TEST['x']), TEST),
       'INT8': score(predict_ort(ort_sess(paths['INT8']), TEST['x']), TEST)}
elems = {'FP32': peak_act_elems(base), 'Pruned FP32': peak_act_elems(FINAL_FP)}; elems['INT8'] = elems['Pruned FP32']
bpe = {'FP32': 4, 'Pruned FP32': 4, 'INT8': 1}
rows = {}
for k in ['cloud_acc', 'cloud_F1', 'cloud_AUC', 'cloud_frac_MAE', 'cloud_sel_acc@.95', 'cloud_cov@.95',
          'cls_top1', 'cls_macroF1', 'cls_sel_acc@.95', 'cls_cov@.95']:
    rows[k] = [res[n][k] for n in paths]
rows['DROP cloud_acc (pts)'] = [res['FP32']['cloud_acc'] - res[n]['cloud_acc'] for n in paths]
rows['DROP cls_top1 (pts)'] = [res['FP32']['cls_top1'] - res[n]['cls_top1'] for n in paths]
rows['Flash MB (file size)'] = [os.path.getsize(paths[n]) / 1e6 for n in paths]
rows['Flash compression x'] = [os.path.getsize(paths['FP32']) / os.path.getsize(paths[n]) for n in paths]
rows['Peak act RAM MB @1B/elem'] = [elems[n] / 1e6 for n in paths]
rows['Peak act RAM MB (native dtype)'] = [elems[n] * bpe[n] / 1e6 for n in paths]
rows['RAM compression x (vs FP32 native)'] = [elems['FP32'] * 4 / (elems[n] * bpe[n]) for n in paths]
rows['CPU latency ms/tile (1 thread)'] = [latency_ms(paths[n]) for n in paths]
TABLE = pd.DataFrame(rows, index=list(paths)).T.round(3)
print(TABLE.to_string())
print('\nPeak RAM check (INT8, 1 B/elem): %.3f MB  -> <4.2 MB: %s | <1.28 MB: %s' %
      (elems['INT8'] / 1e6, elems['INT8'] / 1e6 < 4.2, elems['INT8'] / 1e6 < 1.28))
print('Method: ONNX graph with static 1x3x160x160 input; at each node sum sizes of all live tensors (inputs+outputs+pending skips); max over nodes.')

In [ ]:
# 11) Temperature scaling (T_cloud, T_class) on VAL using INT8 logits + reliability tables.
from scipy.optimize import minimize_scalar
FSESS = ort_sess('/content/model_int8.onnx')
ov, ot = predict_ort(FSESS, VAL['x']), predict_ort(FSESS, TEST['x'])
def nll_bin(z, y, T): p = np.clip(sig(z / T), 1e-7, 1 - 1e-7); return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()
def nll_cat(z, y, T): return -np.log(np.clip(softmax(z / T)[np.arange(len(y)), y], 1e-9, 1)).mean()
mv_ = VAL['yk'].numpy() >= 0
T_CLOUD = float(minimize_scalar(lambda T: nll_bin(ov[0][:, 0], VAL['yc'].numpy(), T), bounds=(.05, 10), method='bounded').x)
T_CLASS = float(minimize_scalar(lambda T: nll_cat(ov[2][mv_], VAL['yk'].numpy()[mv_], T), bounds=(.05, 10), method='bounded').x)
print(f'T_cloud={T_CLOUD:.3f}  T_class={T_CLASS:.3f}')

def reliab(conf, ok, title):
    rows = []
    for t in (.5, .7, .9, .95, .99):
        s = conf >= t
        rows.append(dict(conf_ge=t, n=int(s.sum()), coverage_pct=round(100 * s.mean(), 1), actual_acc_pct=round(100 * ok[s].mean(), 2) if s.any() else np.nan))
    print('\n' + title); print(pd.DataFrame(rows).to_string(index=False))

def reliab_all(o, ev, Tc, Tk, tag):
    p = sig(o[0][:, 0] / Tc); y = ev['yc'].numpy(); reliab(np.maximum(p, 1 - p), (p > .5).astype(int) == y, f'CLOUD head {tag}')
    m = ev['yk'].numpy() >= 0; pr = softmax(o[2][m] / Tk); reliab(pr.max(1), pr.argmax(1) == ev['yk'].numpy()[m], f'CLASS head {tag}')
reliab_all(ot, TEST, 1, 1, 'TEST, T=1 (before)')
reliab_all(ot, TEST, T_CLOUD, T_CLASS, 'TEST, temperature-scaled (after)')
print('\nIf actual_acc at conf>=0.95 is below 95%, raise cloud_drop_conf / conf_t in config.json until the row reads >=95%.')

In [ ]:
# 12) Write config.json (all thresholds live here; edit salience_weights freely).
EMB_D = int(ov[3].shape[1])
CFG_DICT = dict(tile_size=S, S=S, T_cloud=T_CLOUD, T_class=T_CLASS, class_names=CLASS_NAMES, salience_weights=SALIENCE,
                cloud_drop_conf=0.95, conf_t=0.95, embedding_dim=EMB_D, jpeg_quality=70,
                frame_score_weights=[0.6, 0.4], budget_bps=256000, pass_seconds=600)
json.dump(CFG_DICT, open('/content/config.json', 'w'), indent=2)
print(json.dumps({k: v for k, v in CFG_DICT.items() if k != 'salience_weights'}))

In [ ]:
# 13) triage_frame(): tiling, ONNX inference, decision logic, priority scores, refined image.
CFG = json.load(open('/content/config.json')); SESS = ort_sess('/content/model_int8.onnx')
cos_dist = lambda a, b: float(np.clip(1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9), 0, 1))

def tile_frame(img):                                    # HxWx3 uint8 -> (R*C,S,S,3) with edge padding, plus grid size
    H, W = img.shape[:2]; R, C = math.ceil(H / S), math.ceil(W / S)
    pad = np.pad(img, ((0, R * S - H), (0, C * S - W), (0, 0)), mode='edge')
    return pad.reshape(R, S, C, S, 3).transpose(0, 2, 1, 3, 4).reshape(R * C, S, S, 3), R, C

def triage_frame(img, filename='frame.png', sess=None, cfg=None, ref=None):
    """img: HxWx3 uint8 RGB. ref: optional {(row,col): embedding} from the previous pass. Returns a result dict."""
    sess, cfg = sess or SESS, cfg or CFG; t0 = time.perf_counter()
    H, W = img.shape[:2]; tiles, R, C = tile_frame(img); xin = tiles.transpose(0, 3, 1, 2).astype(np.float32) / 255
    o = [sess.run(OUT, {'x': xin[i:i+32]}) for i in range(0, len(xin), 32)]
    cl, fl, ck, em = [np.concatenate([a[k] for a in o]) for k in range(4)]
    p = sig(cl[:, 0] / cfg['T_cloud']); cconf = np.maximum(p, 1 - p); cfrac = sig(fl[:, 0])          # step 1
    pr = softmax(ck / cfg['T_class']); kconf, kcls = pr.max(1), pr.argmax(1)
    sal = pr @ np.array([cfg['salience_weights'][c] for c in cfg['class_names']])
    clear = np.where(p <= 0.5)[0]; nov = np.zeros(len(p))
    for t in range(len(p)):                                                                           # novelty
        r, c = divmod(t, C)
        if ref is not None and (r, c) in ref: nov[t] = cos_dist(em[t], ref[(r, c)])
        else:
            others = [j for j in clear if j != t]
            if others: nov[t] = cos_dist(em[t], em[others].mean(0))
    score_t = sal * nov * (1 - cfrac)                                                                 # step 5
    tiles_out, alerts, crops = [], [], []
    for t in range(len(p)):
        r, c = divmod(t, C); box = [c * S, r * S, min((c + 1) * S, W), min((r + 1) * S, H)]
        if p[t] > 0.5 and cconf[t] >= cfg['cloud_drop_conf']: act = 'discard'                         # step 2
        elif p[t] > 0.5: act = 'hold'                                                                 # step 3
        elif kconf[t] >= cfg['conf_t']: act = 'alert'                                                 # step 4
        else: act = 'queue_crop'
        sc = 0.0 if act == 'discard' else float(score_t[t])
        rec = dict(row=r, col=c, bbox=box, action=act, p_cloud=float(p[t]), cloud_frac=float(cfrac[t]),
                   cls=CLASS_NAMES[kcls[t]], conf=float(kconf[t]), score=sc); tiles_out.append(rec)
        if act == 'alert':
            alerts.append(dict(f=filename, r=r, c=c, bbox=box, cls=rec['cls'], conf=round(rec['conf'], 3), score=round(sc, 4)))
        elif act in ('queue_crop', 'hold'):
            buf = io.BytesIO(); Image.fromarray(img[box[1]:box[3], box[0]:box[2]]).save(buf, 'JPEG', quality=cfg['jpeg_quality'])
            crops.append(dict(f=filename, r=r, c=c, bbox=box, reason='ambiguous_cloud' if act == 'hold' else 'low_class_conf',
                              score=sc, jpeg=buf.getvalue()))
    kept = [x['score'] for x in tiles_out if x['action'] != 'discard']
    raw = 0.6 * max(kept) + 0.4 * float(np.mean(kept)) if kept else 0.0                               # step 6 (pre-normalisation)
    refined, annot = img.copy(), Image.fromarray(img.copy()); dr = ImageDraw.Draw(annot)
    colour = {'alert': (0, 255, 0), 'queue_crop': (255, 255, 0), 'hold': (255, 140, 0)}
    for x in tiles_out:
        b = x['bbox']
        if x['action'] == 'discard': refined[b[1]:b[3], b[0]:b[2]] = 0
        else:
            if x['action'] in colour: dr.rectangle(b, outline=colour[x['action']], width=3)
    for x in tiles_out:                                                                               # blank discarded tiles in annotated copy too
        if x['action'] == 'discard': dr.rectangle(x['bbox'], fill=(0, 0, 0))
    cnt = pd.Series([x['action'] for x in tiles_out]).value_counts().to_dict()
    return dict(filename=filename, raw_priority=raw, tiles=tiles_out, alerts=alerts, crops=crops, refined=refined,
                annotated=np.asarray(annot), emb={(x['row'], x['col']): em[i] for i, x in enumerate(tiles_out)},
                counts=cnt, infer_ms=(time.perf_counter() - t0) * 1000, raw_bytes=H * W * 3)

def normalise_priorities(results):                      # frame priority normalised to 0-1 across the pass
    m = max([r['raw_priority'] for r in results] + [1e-9])
    for r in results: r['priority_score'] = r['raw_priority'] / m
    return results
print('triage_frame ready')

In [ ]:
# 14) Test on demo frames (EuroSAT mosaics with big clouds), optional real frames, user images; write JSON + refined images.
os.makedirs('/content/frames', exist_ok=True); os.makedirs('/content/output', exist_ok=True); os.makedirs('/content/sat_images', exist_ok=True)
seed_all(123)
def demo_frame(k, rows=3, cols=4):                      # stitch rows*cols test tiles into one frame, apply one large cloud field
    ids = torch.randint(0, len(Xte), (rows * cols,), device=dev)
    t = to160(Xte[ids]); canvas = t.view(rows, cols, 3, S, S).permute(2, 0, 3, 1, 4).reshape(1, 3, rows * S, cols * S)
    canvas, _ = add_clouds(canvas, clean_p=0.15, octaves=(1, 2, 4, 8, 16, 32), snow_p=0.3)
    arr = (canvas[0].permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)
    Image.fromarray(arr).save(f'/content/frames/stitched_{k:03d}.png')
N_DEMO = 30
for k in range(1, N_DEMO + 1): demo_frame(k)
if REAL:                                                # two frames stitched from real 38-Cloud validation tiles
    for k in range(2):
        ids = np.random.RandomState(k).choice(len(Rva), 12, replace=False)
        t = F.interpolate(Rva[ids].permute(0, 3, 1, 2).float() / 255, size=(S, S), mode='bilinear', align_corners=False)
        cv = t.view(3, 4, 3, S, S).permute(2, 0, 3, 1, 4).reshape(3, 3 * S, 4 * S)
        Image.fromarray((cv.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)).save(f'/content/frames/real38_{k+1:03d}.png')
files = sorted(glob.glob('/content/frames/*.png')) + sorted(sum([glob.glob(f'/content/sat_images/*.{e}') for e in ('png', 'jpg', 'jpeg', 'tif', 'tiff')], []))
print(len(files), 'frames (put your own RGB images in /content/sat_images/ and re-run this cell)')

RESULTS = []
for f in files:
    img = np.asarray(Image.open(f).convert('RGB'))
    if max(img.shape[:2]) > 6000: print('large image', f, img.shape, '- tiling may take a while')
    RESULTS.append(triage_frame(img, os.path.basename(f)))
normalise_priorities(RESULTS)
for r in RESULTS:
    Image.fromarray(r['refined']).save(f"/content/output/refined_{r['filename']}")
    Image.fromarray(r['annotated']).save(f"/content/output/annotated_{r['filename']}")
final = {'images': [{'filename': r['filename'], 'priority_score': round(float(r['priority_score']), 4)}
                    for r in sorted(RESULTS, key=lambda r: -r['priority_score'])]}
json.dump(final, open('/content/output/priority.json', 'w'))
print(json.dumps(final)[:400], '...')
print('example tile decisions for', RESULTS[0]['filename'], RESULTS[0]['counts'])
print(pd.DataFrame([{k: (round(v, 3) if isinstance(v, float) else v) for k, v in t.items() if k != 'bbox'} for t in RESULTS[0]['tiles']]).to_string(index=False))
from IPython.display import display
display(Image.fromarray(np.concatenate([np.asarray(Image.open(files[0]).convert('RGB')), RESULTS[0]['annotated']], 1)).resize((1280, 480)))

In [ ]:
# 15) Pass simulator: 256 kbps x 600 s = 19.2 MB budget; alerts first, then crops by priority until the budget is used.
def simulate_pass(results, bps=256000, seconds=600):
    budget = bps * seconds / 8; rate = bps / 8                                   # bytes, bytes/s
    raw = sum(r['raw_bytes'] for r in results)
    alerts = sorted([a for r in results for a in r['alerts']], key=lambda a: -a['score'])
    ab = [len(json.dumps(a, separators=(',', ':')).encode()) for a in alerts]
    crops = sorted([c for r in results for c in r['crops']], key=lambda c: -c['score'])
    sent_a = 0; used = 0
    for b in ab:
        if used + b > budget: break
        used += b; sent_a += 1
    sent_c, cb = 0, 0
    for c in crops:
        if used + len(c['jpeg']) > budget: break
        used += len(c['jpeg']); cb += len(c['jpeg']); sent_c += 1
    cum, first = 0.0, None                                                       # time to first alert = on-board inference so far + airtime
    for r in results:
        cum += r['infer_ms'] / 1000
        if r['alerts']: first = cum + len(json.dumps(r['alerts'][0], separators=(',', ':')).encode()) / rate; break
    print(f'Frames processed        : {len(results)}')
    print(f'Raw imagery (uncompr.)  : {raw/1e6:.2f} MB  (naive downlink would take {raw/rate:.0f} s vs {seconds} s pass)')
    print(f'Downlink budget         : {budget/1e6:.2f} MB')
    print(f'JSON alerts sent        : {sent_a}/{len(alerts)}  ({sum(ab[:sent_a])} bytes)')
    print(f'Crops queued / sent     : {len(crops)} / {sent_c}  ({cb/1e6:.3f} MB sent of {sum(len(c["jpeg"]) for c in crops)/1e6:.3f} MB queued)')
    print(f'Total bytes sent        : {used} ({used/1e6:.3f} MB, {used/rate:.1f} s of airtime)')
    print(f'Bandwidth reduction     : {100*(1-used/raw):.2f} %   (target >= 80 %)')
    print('Time to first alert     : ' + (f'{first:.2f} s' if first is not None else 'no alert raised'))
    print(f'Mean on-board latency   : {np.mean([r["infer_ms"] for r in results]):.1f} ms/frame')
simulate_pass(RESULTS)

In [ ]:
# 16) Zip the deployable artefacts for download.
with zipfile.ZipFile('/content/deliverable.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/content/model_int8.onnx', 'model_int8.onnx'); z.write('/content/config.json', 'config.json')
print('deliverable.zip:', os.path.getsize('/content/deliverable.zip') / 1e6, 'MB (also see /content/output for priority.json + refined images)')
try:
    from google.colab import files as _f; _f.download('/content/deliverable.zip')
except Exception: pass

## Troubleshooting & honest caveats
**torch-pruning API**: newer versions use `pruning_ratio=`, older ones `ch_sparsity=` (handled by try/except). If the tracer fails on MobileNetV3
(`AttributeError`/shape errors around SqueezeExcitation), upgrade (`pip install -U torch-pruning`), or reduce `iterative_steps`, or add the SE convs
(`block[...].fc1/fc2`) to `ignored_layers`. Pruned channel counts are arbitrary; add `round_to=8` to MetaPruner if your version supports it.
Head `Linear` layers are all in `ignored_layers` (their outputs stay fixed, inputs shrink with the pruned embedding).

**ONNX / quantization warnings**: "Please consider to run pre-processing before quantization" is cosmetic (we run `quant_pre_process` when possible).
`dynamo=False` forces the legacy exporter (torch >= 2.5 defaults to dynamo, which may ignore opset 13). Messages about
"Missing/unsupported scale" or `HardSigmoid` left in float are normal: those ops stay float between QDQ pairs. If `quantize_static` fails
with a type/shape error, re-run with `pip install -U onnxruntime onnx`.

**RAM estimate** is an analytic lower-bound-style model (1 byte/element, no weights, no operator-internal scratch buffers, no tiling/in-place fusion);
an MCU runtime such as TFLM/ORT-mobile adds scratch + arena overhead, so keep >2x margin vs 4.2 MB.

**Not verified by me (no GPU/Colab access when writing)**: the full notebook was syntax-checked only, not executed. Specifically untested:
EuroSAT download, torch-pruning on MobileNetV3-small with SE blocks, whether the <2-point rule is met at every prune step, INT8 accuracy,
the 38-Cloud branch (`cache.npz`), and the reported bandwidth reduction numbers. Novelty (cosine distance of 576-dim pooled features) is typically
small (<0.3), so priority scores are compressed toward 0; frame priorities are normalised by the max so ordering is unaffected.
Real-image generalisation (different GSD, snow vs cloud) depends on the augmentations; check results on your own images.
Deviation from spec: 38-Cloud holds out 4 scenes for validation plus 2 extra scenes for test (so test metrics stay unbiased).